# 06: 多方法注释与交叉比对

对每个 Leiden 簇并行运行多种标注方法，交叉比对结果。
**多方法并跑不是冗余——不同独立方法的共识最能提高置信度**，
分歧标记需要 PI 重点复核的簇。

本 notebook 产出：
- 各方法独立标注列（`cell_type_{method}_v1`）
- 成对混淆矩阵热图 + Cohen's kappa 表
- 每簇 LLM 综合判决 markdown（需配 API key）
- PI 最终标注列 `cell_type_final_v1`

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：05（多分辨率 Leiden 聚类），读 `05_clustered_v*.h5ad`
- **下游**：06c（亚群 Subset 重分析）/ 07（下游分析），产出 `06_annotated_v*.h5ad`

### 为什么要迭代回跑？
注释质量直接影响下游逐簇分析和亚群重分析的准确性。如果在 06c（亚群分析发现注释不合理）、
07（跨病种比较时发现标签粒度不对）或逐簇报告中发现问题，可能需要：
- 换用不同的 Leiden 分辨率的列（修改 `LEIDEN_COL`）
- 增加或替换标记物 CSV（修改 `MARKER_CSV`）
- 调整 LLM 模型选择（修改 `MLLM_MODELS` / `MLLM_CONSENSUS_THRESHOLD` / `VERDICT_MODEL_TIER`）
- 在 PI 手动标注区修改 `marker_assignments` 或 `pi_decisions` 的标签
- 换用 05 的另一个版本（不同 Leiden 分辨率组合）

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `05_clustered_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `06_annotated_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `LEIDEN_COL`、`MARKER_CSV`、
   `LLM_MODELS` 等），然后 Run All 重跑全部 cell。

> 原始构思（PI）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建、
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

**起点衔接——本 notebook 默认从 05 的最稳定版本出发。**

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH  — 05（已聚类）输出文件路径
# OUTPUT_PATH    — 本 stage 产出 checkpoint 路径
# MARKER_CSV     — 标记物知识库 CSV（供 dotplot + 基因集评分）
# LEIDEN_COL     — 用作簇标签的 obs 列
# RANDOM_SEED    — 随机种子，确保可复现

UPSTREAM_PATH = "results/05_clustered_v1.h5ad"
OUTPUT_PATH   = "results/06_annotated_v1.h5ad"

MARKER_CSV  = "references/markers/gastric_TEST_markers.csv"  # 测试夹具，PI 后续替换为真实 marker 库
LEIDEN_COL  = "leiden_res_0.6"
RANDOM_SEED = 42

# === LLM 配置（读 vault 根 .env 的 LLM_GROUP{N}_* schema）===
# LLM_GROUP  — 使用第几个 LLM group（对应 .env 的 LLM_GROUP{N}_*）。
#              None = 自动用 LLM_DEFAULT_GROUP（当前 .env 里设为 1）。
#              每个 group 包含 1 个 provider + base_url + api_key + 3 档模型。
#              多 group 可用时，mLLMCelltype 共识会自动使用多个端点。
LLM_GROUP = None  # None → 自动取 LLM_DEFAULT_GROUP；手工指定：LLM_GROUP = 1

# --- LLM 注释（mLLMCelltype 多模型共识）---
# mLLMCelltype 同时调用多个 LLM 模型独立注释每簇，然后多数票投票产生共识标签。
# 当模型间分歧 > 阈值时自动触发多轮讨论（discussion rounds），直到达成一致。
MLLM_ENABLED = True
MLLM_MODELS = None          # None → 从 .env LLM_GROUP 自动构建模型列表
                             # 或手动指定如 ["claude-3-5-haiku-20241022", "deepseek-chat"]
MLLM_CONSENSUS_THRESHOLD = 0.7   # 标签比例需 >=70% 才算"达成共识"
MLLM_ENTROPY_THRESHOLD = 0.3     # 标签分布熵需 <=0.3 才算"意见集中"
MLLM_MAX_DISCUSSION_ROUNDS = 2   # 多模型讨论最多 2 轮

# --- LLM 证据汇总判决 ---
#   "sonnet" — 平衡，适合证据汇总判决（推理-成本平衡）
#   "opus"  — 最强推理，适合争议簇复核（贵、慢，按需启用）
VERDICT_MODEL_TIER = "sonnet"  # 证据汇总用

# === scANVI 参考 atlas（不存在则优雅跳过） ===
REFERENCE_ATLAS_PATH = ""  # 留空跳过；填入 .h5ad 路径启用


In [ ]:
# === setup：sys.path + 导入 + 加载上游 adata + 标记物知识库 ===

# 1. 确保框架 src/ 在 sys.path 上，CWD 为项目根目录
import sys, os, gc
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures/06_verdicts", exist_ok=True)
os.makedirs("results/figures/06_sankey", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# 2. 导入（scanpy 原生 API + 框架函数）
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import datetime, warnings

from scrna_integration import load_markers
from scrna_integration.scorers import annotation_concordance

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)

# 3. 加载上游 05 产出
print("加载上游:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"obsm 键: {list(adata.obsm.keys())}")
print(f"leiden 列 '{LEIDEN_COL}': "
      f"{adata.obs[LEIDEN_COL].nunique()} 个簇")

# 4. 加载标记物知识库（marker CSV 不存在时优雅跳过）
if os.path.exists(MARKER_CSV):
    markers = load_markers(MARKER_CSV)
    print(f"标记物库: {MARKER_CSV}")
    print(f"  细胞类型数: {len(markers)}")
    for ct, genes in list(markers.items())[:5]:
        print(f"  {ct}: {genes}")
    if len(markers) > 5:
        print(f"  ... 共 {len(markers)} 种细胞类型")

    # 展平为所有标记基因列表（用于 dotplot）
    all_marker_genes = sorted(set(g for glist in markers.values() for g in glist))
    # 只保留在 adata 中实际存在的基因
    available_markers = [g for g in all_marker_genes if g in adata.var_names]
    missing = set(all_marker_genes) - set(available_markers)
    if missing:
        print(f"  数据中不存在的标记基因（跳过）: {sorted(missing)}")
    print(f"  可用标记基因: {len(available_markers)}/{len(all_marker_genes)}")
else:
    print(f"marker CSV 不存在 ({MARKER_CSV})，跳过标记物知识库加载")
    markers = {}
    available_markers = []

In [ ]:
# === LLM 响应解析 helper（放在这里方便学生看到定义） ===
# 从 LLM 的文本回复中提取 JSON 标注字典，容忍 ```json``` 代码围栏和 thinking 块
import re as _re

def _extract_json_from_llm_response(raw_text: str, expected_clusters: list):
    """从 LLM 原始回复中提取 JSON 标注字典。

    容忍多种格式：纯 JSON、```json ... ``` 代码块、键不带引号等。
    返回 {cluster_id: cell_type} 或空 dict。
    """
    import json as _json

    if not raw_text:
        return {}

    # 策略 1：匹配 ```json ... ``` 代码块
    _match = _re.search(r'```(?:json)?\s*\n?(.*?)\n?```', raw_text, _re.DOTALL)
    if _match:
        _candidate = _match.group(1).strip()
    else:
        # 策略 2：找第一个 { 和最后一个 } 之间的内容
        _start = raw_text.find("{")
        _end = raw_text.rfind("}")
        if _start >= 0 and _end > _start:
            _candidate = raw_text[_start:_end + 1]
        else:
            return {}

    try:
        _result = _json.loads(_candidate)
        if isinstance(_result, dict):
            # 键可能是 int 或 str，统一为 str
            return {str(k): str(v) for k, v in _result.items() if v}
    except (_json.JSONDecodeError, ValueError):
        pass

    return {}

## 方法 1：标记物 dotplot（PI 手动标注）

用已有标记物知识库画 dotplot，每个簇表达哪些标记物一目了然。
**为什么先做这个？** 让 PI 在受自动方法影响前建立自己的判断，避免锚定偏差。
（也可以后做——方法顺序不影响结果，PI 自由选择。）

### 怎么看 dotplot？
- **横轴**：标记基因；**纵轴**：簇
- **颜色深浅**：该基因在该簇的平均表达量
- **圆点大小**：该簇中表达该基因的细胞百分比
- **判断规则**：某个簇对某类细胞的全部标记基因都表达（大圆点 + 深色）
  → 该簇很可能是该细胞类型；只表达个别标记基因 → 可能不是或需更多证据

PI 浏览此图后在下方的 `marker_assignments` 字典中填写每个簇的细胞类型。

In [ ]:
# 标记物 dotplot——每个簇 x 每个标记基因的（表达百分比 + 平均表达量）
if available_markers and LEIDEN_COL in adata.obs.columns:
    sc.pl.dotplot(
        adata, var_names=available_markers, groupby=LEIDEN_COL,
        dendrogram=True, standard_scale="var",
        title=f"Canonical marker dotplot ({LEIDEN_COL})",
        save="_06_dotplot.png",
    )
    # scanpy 默认保存到 figures/，移动到 results/figures/
    src = "figures/dotplot__06_dotplot.png"
    dst = "results/figures/06_dotplot.png"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"dotplot 已保存: {dst}")
    plt.close("all")
else:
    print("无可用的标记基因或缺少 leiden 列，跳过 dotplot")

In [ ]:
# === Marker Dotplot（按细胞类型分组）===
# 每个细胞类型作为一组（而非把所有 marker 混在一起），
# 组间用分隔线区分，更直观地展示"该簇是否符合某细胞类型的完整 marker profile"。
# 为什么保留 flat dotplot（上方 cell）？flat 版本展示全部 marker 的相对排序，
# 分组版本展示类型内 marker 一致性——两个视角互补。

import shutil
if markers and available_markers:
    # 构建分组 dict 供 sc.pl.dotplot 使用
    _var_names_grouped = {}
    for ctype, genes in markers.items():
        _present = [g for g in genes if g in adata.var_names]
        if _present:
            _var_names_grouped[ctype] = _present

    if _var_names_grouped:
        print(f"Dotplot 分组: {len(_var_names_grouped)} 组, 共 {len(available_markers)} 个标记基因")
        sc.pl.dotplot(
            adata,
            var_names=_var_names_grouped,
            groupby=LEIDEN_COL,
            standard_scale="var",
            dendrogram=True,
            show=True,
            save="_06_dotplot_grouped.png",
        )
        # scanpy 默认保存到 figures/，移动到 results/figures/
        _src = "figures/dotplot__06_dotplot_grouped.png"
        _dst = "results/figures/06_dotplot_grouped.png"
        if os.path.exists(_src):
            shutil.move(_src, _dst)
            print(f"  保存: {_dst}")
        plt.close("all")
    else:
        print("⚠️ 无可用标记基因，跳过分组 dotplot")


In [ ]:
# === PI 手动标注区 ===
# PI 查看上方 dotplot 后，在下方字典中为每个簇 ID 填入细胞类型。
# 示例（基于 Nowicki 数据——PI 需根据实际 dotplot 修改）：
marker_assignments = {
    # "0": "B_cell",
    # "1": "T_cell",
    # "2": "myeloid",
    # ...  PI 逐簇填入
}

if marker_assignments:
    adata.obs["cell_type_marker_v1"] = (
        adata.obs[LEIDEN_COL].astype(str).map(marker_assignments)
    )
    n_assigned = adata.obs["cell_type_marker_v1"].notna().sum()
    print(f"marker 标注: {n_assigned}/{adata.n_obs} 细胞已标注")
else:
    # PI 暂未填写，预建空列保持结构完整
    adata.obs["cell_type_marker_v1"] = np.nan
    adata.obs["cell_type_marker_v1"] = (
        adata.obs["cell_type_marker_v1"].astype("category")
    )
    print("marker 标注: PI 暂未填写，已预建 cell_type_marker_v1 空列")

## 方法 2：mLLMCelltype 多模型共识注释

用 mLLMCelltype 的 `interactive_consensus_annotation()` 同时调用多个 LLM 模型
独立注释每簇，取多数票作为共识标签。

**为什么用 mLLMCelltype？**
- **多模型共识**：不同模型的独立判断互相验证，共识标签置信度远高于单模型
- **自动讨论机制**：模型间意见分歧时自动触发多轮讨论（discussion rounds），
  模拟"专家会诊"——各模型说明自己的推理并参考他人观点，直到达成共识
- **配置从 .env 自动读取**：无需手动管理 api_key 列表，框架自动构建
- **组织上下文**：传入 `tissue_type="human gastric mucosa"` 让 LLM 结合胃粘膜背景知识

### 共识参数怎么调？
- `consensus_threshold=0.7`：某标签获得 >=70% 模型投票即认为"达成共识"
- `entropy_threshold=0.3`：标签分布熵 <=0.3（低熵 = 意见集中）
- `max_discussion_rounds=2`：最多 2 轮讨论，避免无限辩论
- 调低阈值 → 更宽松（更快但可能遗漏分歧）；调高 → 更严格（更慢但更可靠）

### 怎么看注释结果？
每簇输出模型投票分布和最终共识标签。各模型投票不一致的簇自动标记为
"争议簇"（low-confidence），供 PI 重点复核。

> 技术说明：为什么从 requests 直连改回 mLLMCelltype？
> 之前因本地网关 thinking 块不兼容从 mLLMCelltype 切换到 requests 直连。
> 现在切换回来是因为：(1) mLLMCelltype 的 interactive_consensus_annotation
> 提供多模型讨论机制，比手写多数票投票更稳健；(2) 模型管理和错误重试由包内部处理，
> notebook 代码更简洁；(3) 上游已修复 thinking 块兼容性。


In [ ]:
# === 方法 2: mLLMCelltype 多模型共识注释 ===
# 从 .env LLM_GROUP 自动构建模型列表 → 每簇独立注释 → 多模型投票共识
# 为什么用 interactive_consensus_annotation 而非手写 requests？
# (1) 内置多模型讨论机制——模型间自动交换观点后再投票
# (2) 自动管理重试和超时——单模型失败不中断整体流程
# (3) notebook 代码量从 ~150 行压缩到 ~50 行——学生更容易读懂

if MLLM_ENABLED:
    try:
        from mllmcelltype import interactive_consensus_annotation

        # 从 .env 构建 mLLMCelltype 需要的配置
        from scrna_integration.llm_config import load_llm_group_config, get_active_groups
        _active_groups = get_active_groups(project_root=_root)

        if _active_groups:
            # 构建 api_keys 和 base_urls 字典
            _api_keys = {}
            _base_urls = {}
            _model_list = []

            for gid in _active_groups:
                _cfg = load_llm_group_config(group=gid, project_root=_root)
                if _cfg:
                    _provider = _cfg.get("provider", "")
                    _key = _cfg.get("api_key", "")
                    _url = _cfg.get("base_url", "")
                    # 用 haiku 档模型做注释（便宜、快）
                    _model = _cfg.get("models", {}).get("haiku", "")

                    if _model and _key:
                        # mLLMCelltype 用模型名前缀自动路由 provider
                        _model_list.append(_model)
                        _api_keys[_model] = _key
                        if _url:
                            _base_urls[_model] = _url

            # 允许手动覆盖模型列表（PARAMS 中 MLLM_MODELS 非 None 时生效）
            if MLLM_MODELS:
                _model_list = MLLM_MODELS
                # 校验覆盖模型是否有对应 API key（.env LLM_GROUP 配置是否完整）
                _missing_keys = [m for m in _model_list if m not in _api_keys]
                if _missing_keys:
                    print(f"⚠️ MLLM_MODELS 中以下模型无对应 API key（检查 .env LLM_GROUP 配置）:")
                    for m in _missing_keys:
                        print(f"    {m}")
                    _model_list = [m for m in _model_list if m in _api_keys]
                    if not _model_list:
                        print("  → 所有指定模型均无 key，跳过 mLLMCelltype")
                    else:
                        print(f"  → 使用有 key 的子集: {_model_list}")

            if _model_list:
                print(f"mLLMCelltype 模型列表: {_model_list}")
                print(f"开始多模型共识注释（{len(_model_list)} 个模型）...")

                # 准备输入：需要 rank_genes_groups 结果（mLLMCelltype 用它提取每簇标记基因）
                if "rank_genes_06" not in adata.uns:
                    sc.tl.rank_genes_groups(
                        adata, groupby=LEIDEN_COL, method="wilcoxon",
                        n_genes=30, key_added="rank_genes_06",
                    )

                # 调用 mLLMCelltype
                _result = interactive_consensus_annotation(
                    adata,
                    cluster_key=LEIDEN_COL,
                    models=_model_list,
                    api_keys=_api_keys,
                    base_urls=_base_urls if _base_urls else None,
                    tissue_type="human gastric mucosa",
                    consensus_threshold=MLLM_CONSENSUS_THRESHOLD,
                    entropy_threshold=MLLM_ENTROPY_THRESHOLD,
                    max_discussion_rounds=MLLM_MAX_DISCUSSION_ROUNDS,
                )

                # 提取结果（mLLMCelltype 返回 AnnotationResult 对象或 dict）
                if hasattr(_result, "cell_types") and _result.cell_types:
                    _llm_map = _result.cell_types  # {cluster_id: label}
                    adata.obs["cell_type_llm_v1"] = (
                        adata.obs[LEIDEN_COL].astype(str).map(_llm_map).astype("category")
                    )
                    print(f"✓ mLLMCelltype 注释完成: {len(_llm_map)} 簇")
                    for cid, label in sorted(_llm_map.items(), key=lambda x: int(x[0])):
                        print(f"    Cluster {cid}: {label}")
                elif isinstance(_result, dict):
                    # 兼容旧版返回格式（纯 dict）
                    adata.obs["cell_type_llm_v1"] = (
                        adata.obs[LEIDEN_COL].astype(str).map(_result).astype("category")
                    )
                    print(f"✓ mLLMCelltype 注释完成 (dict 格式): {len(_result)} 簇")
                else:
                    print(f"⚠️ mLLMCelltype 返回格式异常（类型: {type(_result).__name__}），跳过")
            else:
                print("⚠️ 无可用模型（.env 缺少 haiku 档模型配置），跳过 mLLMCelltype 注释")
                print("  提示：确认 .env 中 LLM_GROUP{N}_MODEL_HAIKU 已设置")
        else:
            print("⚠️ 无活跃 LLM group（.env 未配置），跳过 mLLMCelltype 注释")
            print("  提示：在 AI-OS vault 根 .env 文件中配置 LLM_GROUP{N}_*")

    except ImportError:
        print("⚠️ mLLMCelltype 未安装，跳过。安装命令：pip install mllmcelltype")
    except Exception as e:
        print(f"⚠️ mLLMCelltype 运行异常: {e}")
        import traceback
        traceback.print_exc()
        print("  -> fallback: 可用 claude -p 手动逐簇注释")
else:
    print("MLLM_ENABLED=False，跳过 mLLMCelltype 注释")

# 确保列存在（即使跳过——保持下游列结构完整）
if "cell_type_llm_v1" not in adata.obs.columns:
    adata.obs["cell_type_llm_v1"] = pd.Categorical([np.nan] * adata.n_obs)


## 方法 3：基因集评分

用 `sc.tl.score_genes` 对每类标记基因集合做评分，得到每个细胞相对于每个
细胞类型的连续得分（`obs["score_{celltype}"]`）。

**为什么做基因集评分？** 评分提供了连续性证据——一个簇可能同时高表达多种
细胞类型的标记，说明该簇可能是过渡态或混合群体。评分不直接作为独立标签，而是
作为交叉比对的补充证据：当多个方法对同一簇的标签有分歧时，看该簇对哪种细胞类型的
评分更高，辅助裁决。

### 怎么看评分图？
下方 UMAP 图中，颜色深浅 = 该细胞对该细胞类型的基因集评分（0 到高值）。
某个簇整体颜色深 → 该簇高表达该类标记物 → 支持该细胞类型标注。

In [ ]:
# 对每个细胞类型做基因集评分（sc.tl.score_genes）
# 原理：标记基因平均表达 - 随机参考基因平均表达，产生连续得分
# 用 score_genes 而非 AUCell：scanpy 原生，零额外依赖，学生直接看懂

_score_cols = []
for ct, gene_list in markers.items():
    # 只保留数据中实际存在的基因
    _present = [g for g in gene_list if g in adata.var_names]
    if len(_present) < 2:
        print(f"  {ct}: 可用标记基因 <2（{len(_present)}），跳过评分")
        continue
    col = f"score_{ct}"
    sc.tl.score_genes(
        adata, gene_list=_present, score_name=col,
        ctrl_size=max(1, min(len(_present), 50)),
    )
    _score_cols.append(col)
    print(f"  {ct}: {len(_present)}/{len(gene_list)} 个基因可用 -> obs['{col}']")

print(f"\n共生成 {len(_score_cols)} 个评分列")

In [ ]:
# 逐簇汇总基因集评分——均值 + 阳性细胞百分比
# 单个细胞的评分有噪声，簇级汇总能更稳健地反映该簇的整体标记物信号

if _score_cols and LEIDEN_COL in adata.obs.columns:
    _records = []
    if not hasattr(adata.obs[LEIDEN_COL], "cat") or not pd.api.types.is_categorical_dtype(adata.obs[LEIDEN_COL]):
        adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype("category")
    for _cid in sorted(adata.obs[LEIDEN_COL].cat.categories):
        _mask = adata.obs[LEIDEN_COL] == _cid
        _row = {"cluster": _cid, "n_cells": _mask.sum()}
        for col in _score_cols:
            _vals = adata.obs.loc[_mask, col]
            _ct_name = col.removeprefix("score_")
            _row[f"{_ct_name}_mean"] = round(float(_vals.mean()), 4)
            _row[f"{_ct_name}_pct_pos"] = round(float((_vals > 0).mean()) * 100, 1)
        _records.append(_row)
    _score_summary = pd.DataFrame(_records)
    print("基因集评分逐簇汇总:")
    try:
        from IPython.display import display as ipy_display
        ipy_display(_score_summary)
    except ImportError:
        print(_score_summary.to_string())
    _score_summary.to_csv("results/figures/06_gene_set_scores.csv", index=False)
    print("\n已保存: results/figures/06_gene_set_scores.csv")
else:
    print("无评分列或缺少 leiden 列，跳过逐簇汇总")

In [ ]:
# 基因集评分可视化——UMAP 着色
# 每个子图对应一种细胞类型的评分，颜色深浅 = 该细胞对该类型的得分
# PI 借此判断哪些簇对哪种细胞类型的标记物评分最高

if _score_cols and "X_umap" in adata.obsm:
    n = len(_score_cols)
    n_cols = min(3, n)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5.5 * n_cols, 4.5 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, col in enumerate(_score_cols):
        ct_name = col.removeprefix("score_")
        ax = axes[i]
        sc.pl.umap(adata, color=col, ax=ax, show=False,
                   title=ct_name, cmap="viridis",
                   vmin=0, vmax="p99", frameon=False)

    # 隐藏多余子图
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    fig.savefig("results/figures/06_gene_set_scores_umap.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("基因集评分 UMAP 已保存: results/figures/06_gene_set_scores_umap.png")
    plt.close("all")
elif not _score_cols:
    print("无评分列，跳过基因集评分可视化")
else:
    print("obsm 中无 X_umap，跳过 UMAP 着色")

## 方法 4：scANVI 标签迁移（守卫——有参考 atlas 时启用）

**前提**：存在对应疾病系统的有标注参考 atlas（含 `cell_type` obs 列）。
scANVI 同时利用参考数据的已知标签和目标数据自身的无监督结构做半监督训练，
标签迁移比简单 kNN 映射更稳健。

当前胃粘膜没有公认的标注参考，**自动跳过**。
后续若 CELLxGENE Census 或合作者提供有标注的胃参考数据，
在 PARAMS 中填入 `REFERENCE_ATLAS_PATH` 即可启用。

In [ ]:
# scANVI 标签迁移（守卫——REFERENCE_ATLAS_PATH 不存在则优雅跳过）
# 为什么用 scANVI 而非直接 kNN 映射？scANVI 同时用参考标签和
# 目标数据无监督结构做半监督训练，标签迁移比简单 kNN 更稳健。

# === scANVI 标签迁移（已知限制：scvi-tools >= 1.0 API 变更，待适配）===
if REFERENCE_ATLAS_PATH and os.path.exists(REFERENCE_ATLAS_PATH):
    print("\u26a0\ufe0f scANVI 标签迁移功能暂时禁用")
    print("   原因: scvi-tools >= 1.0 的 SCANVI API 不再接受双 AnnData 输入")
    print("   状态: 待适配（需改为 ref+query 合并单 AnnData + unlabeled_category 模式）")
    print("   影响: cell_type_scanvi_v1 列将为 NaN")
    # 以下为旧版 scANVI 代码，保留作为未来适配参考：
    # import scvi
    # print(f"加载参考 atlas: {REFERENCE_ATLAS_PATH}")
    # _ref = sc.read_h5ad(REFERENCE_ATLAS_PATH)
    # print(f"  参考: {_ref.n_obs:,} 细胞 x {_ref.n_vars:,} 基因")
    # _common = adata.var_names.intersection(_ref.var_names)
    # print(f"  共同基因: {len(_common)}")
    # _adata_q = adata[:, _common].copy()
    # _ref_sub = _ref[:, _common].copy()
    # scvi.model.SCANVI.setup_anndata(_adata_q, batch_key="source_dataset")
    # scvi.model.SCANVI.setup_anndata(_ref_sub, batch_key="source_dataset")
    # _model = scvi.model.SCANVI(
    #     _ref_sub, _adata_q,
    #     labels_key="cell_type",
    #     unlabeled_category="Unknown",
    # )
    # _model.train(max_epochs=50, early_stopping=True)
    # _preds = _model.predict(_adata_q)
    # adata.obs["cell_type_scanvi_v1"] = _preds["cell_type"].values
    # adata.obs["cell_type_scanvi_v1_uncertainty"] = (
    #     1.0 - _preds["cell_type"].probabilities.max(axis=1)
    # )
    # adata.uns["scanvi_v1"] = {
    #     "reference_atlas": REFERENCE_ATLAS_PATH,
    #     "method": "scANVI",
    #     "timestamp": datetime.datetime.now().isoformat(),
    # }
    # print(f"scANVI 完成: {adata.obs['cell_type_scanvi_v1'].nunique()} 类")
else:
    _reason = (
        "未配置 REFERENCE_ATLAS_PATH" if not REFERENCE_ATLAS_PATH
        else f"{REFERENCE_ATLAS_PATH} 不存在"
    )
    print("=" * 60)
    print(f"scANVI 标签迁移已跳过——原因: {_reason}")
    print("如需启用：在 PARAMS 中设置 REFERENCE_ATLAS_PATH 为有标注的 .h5ad 路径，")
    print("并确保参考数据包含 cell_type obs 列。")
    print("=" * 60)

# 确保列存在（即使跳过）
if "cell_type_scanvi_v1" not in adata.obs.columns:
    adata.obs["cell_type_scanvi_v1"] = pd.Categorical([np.nan] * adata.n_obs)
if "cell_type_scanvi_v1_uncertainty" not in adata.obs.columns:
    adata.obs["cell_type_scanvi_v1_uncertainty"] = np.nan

## 方法 5（候选，已注释）：CellTypist 预训练分类器

CellTypist 提供多个预训练模型（如 `Immune_All_Low.pkl`、`Developing_Mouse_Brain.pkl` 等）。
**当前注释原因**：PI 的胃/滑膜系统目前没有很好匹配的预训练模型。
当有匹配模型出现时取消下方代码注释即可启用，新列自动进入跨方法比较。

In [ ]:
# === CellTypist（候选——有对应预训练模型时取消注释启用）===
# 前提：pip install celltypist
# 当存在对应组织的 CellTypist 预训练模型时取消注释：
# # import celltypist
# # predictions = celltypist.annotate(
# #     adata, model="Human_Gastric_Atlas.pkl",
# #     majority_voting=True,
# # )
# # adata.obs["cell_type_celltypist_v1"] = (
# #     predictions.predicted_labels["majority_voting"].values
# # )
# # print(f"CellTypist: {adata.obs['cell_type_celltypist_v1'].nunique()} 类")
print(
    "CellTypist 已注释。"
    "当有匹配本组织的预训练模型时取消注释启用。"
)

## 转化状态连续性检测

### 为什么胃粘膜注释不能只给离散标签？

胃上皮细胞存在连续的分化谱系：**主细胞（Chief）→ SPEM → 肠化（IM）**
不是 "A 或 B" 的二元关系。Leiden 聚类虽然把细胞分成了离散的簇，
但某些簇可能落在两个"纯"状态之间的**连续谱**上——这类簇如果强行标注为
单一离散细胞类型，会丢失关键生物学信息。

**检测原理**：对三个已知转化轴，每个轴配一对 declining（下调）和 rising（上调）
标记基因集。如果某个簇对两组 marker 都有明显表达（均值 >0.3），说明该簇
处于**转化中间态**——不应标为终末类型，而应标为 "early SPEM" 或
"Chief→SPEM transitional" 等。

**三个检测轴**（基于胃粘膜已发表文献）：
1. **Chief→SPEM**：PGA3/PGA4/PGC 等主细胞 marker 下降，TFF2/MUC6/CD44 等 SPEM marker 上升
2. **SPEM→IM**：TFF2/MUC6 等 SPEM marker 下降，CDX2/MUC2/VIL1 等肠化 marker 上升
3. **Normal→Atrophy**：ATP4A/ATP4B/GKN1 等壁细胞 marker 下降，TFF1/MUC5AC 等 pit cell marker 上升

检测结果同时存入 `adata.uns["transition_detection_v1"]`，供后续 LLM 证据汇总引用。


In [ ]:
# === 转化状态连续性检测 ===
# 对每簇计算三个转化轴的 declining/rising marker 均值，判断是否处于转化中间态。
# 原理：两个方向的 marker 同时高表达 → 该簇不单纯是某一种终末状态 → 转化中间态
# 为什么不用 pseudotime？pseudotime 需要指定起点，而这里我们关注的是
# "哪几个簇可能不是纯状态"，而非"每个细胞在谱系上的精确位置"。

TRANSITION_AXES = {
    "Chief→SPEM": {
        "declining": ["PGA3", "PGA4", "PGA5", "GIF", "LIPF", "PGC"],
        "rising": ["TFF2", "WFDC2", "MUC6", "CD44", "CFTR", "AQP5"],
    },
    "SPEM→IM": {
        "declining": ["TFF2", "MUC6", "WFDC2"],
        "rising": ["CDX2", "MUC2", "TFF3", "OLFM4", "VIL1", "KRT20"],
    },
    "Normal→Atrophy": {
        "declining": ["ATP4A", "ATP4B", "GKN1", "GKN2"],
        "rising": ["TFF1", "MUC5AC", "CLDN18"],
    },
}

print("===== 转化状态连续性检测 =====\n")
_transition_flags = {}

for axis_name, genes in TRANSITION_AXES.items():
    _declining_present = [g for g in genes["declining"] if g in adata.var_names]
    _rising_present = [g for g in genes["rising"] if g in adata.var_names]

    if len(_declining_present) < 2 or len(_rising_present) < 2:
        print(f"  {axis_name}: 可用标记基因不足（declining={len(_declining_present)}, "
              f"rising={len(_rising_present)}），跳过")
        continue

    print(f"  --- {axis_name} ---")
    print(f"      declining markers: {_declining_present}")
    print(f"      rising markers: {_rising_present}")

    for cl in sorted(adata.obs[LEIDEN_COL].unique(), key=lambda x: int(x)):
        _mask = adata.obs[LEIDEN_COL] == cl

        _dec_expr = adata[_mask][:, _declining_present].X
        _ris_expr = adata[_mask][:, _rising_present].X
        if sp.issparse(_dec_expr):
            _dec_expr = _dec_expr.toarray()
        if sp.issparse(_ris_expr):
            _ris_expr = _ris_expr.toarray()

        _dec_mean = float(_dec_expr.mean())
        _ris_mean = float(_ris_expr.mean())

        # 判断：如果两组 marker 都有明显表达（>0.3），说明处于转化中
        if _dec_mean > 0.3 and _ris_mean > 0.3:
            _ratio = _ris_mean / (_dec_mean + _ris_mean)
            if _ratio < 0.4:
                _stage = "早期"
            elif _ratio < 0.6:
                _stage = "中期"
            else:
                _stage = "晚期"
            print(f"      ⚠️ Cluster {cl}: {axis_name} 转化{_stage} "
                  f"(declining={_dec_mean:.2f}, rising={_ris_mean:.2f}, 进度≈{_ratio:.0%})")
            _transition_flags[str(cl)] = {
                "axis": axis_name,
                "stage": _stage,
                "progress": f"{_ratio:.0%}",
                "declining_mean": f"{_dec_mean:.2f}",
                "rising_mean": f"{_ris_mean:.2f}",
            }
    print()

if _transition_flags:
    print(f"  共 {len(_transition_flags)} 个簇检测到转化信号")
    print("  → 这些簇不应标注为单一离散类型，建议标为 'X→Y transitional' 或 'early SPEM' 等")
    print("  → LLM 判决和 PI 拍板时请参考上述进度信息")
else:
    print("  ✓ 未检测到明显的转化中间态信号")

# 存入 uns 供后续 LLM 证据汇总引用（注意：用 adata.uns 而非依赖 cell 间变量传递）
adata.uns["transition_detection_v1"] = _transition_flags


## 跨方法比较：混淆矩阵热图 + Cohen's kappa + Sankey

对已产生标签的任意两种方法，计算混淆矩阵和 Cohen's kappa，
并用 Sankey 图可视化标签流。

**为什么做跨方法比较？** 这才是多方法并跑的核心价值：
- **一致的方法增强置信度**——两种独立方法给出相同标签 → 可信度高
- **分歧的方法揭示需要 PI 重点复核的簇**——看哪些簇在方法间标签不一致
- **Cohen's kappa 量化一致性**：>0.8 高一致；0.4-0.8 中等一致；<0.4 低一致
- **Sankey 图可视化标签流**——直观展示"方法 A 的 T_cell 被方法 B 拆成了哪些亚型"

### 怎么看这些图？
- **混淆矩阵热图**：行=方法A的标签，列=方法B的标签，颜色深浅=归一化比例。
  对角线亮 = 两方法一致；非对角线亮 = 该标签被方法B重新分类。
- **Cohen's kappa 表**：数值越高越一致，<0.4 的方法对建议 PI 重点复核。
- **Sankey 图**：流向矩阵保存在 `results/figures/06_sankey/`，供外部工具绘制。

In [ ]:
# 收集所有已产生的标注列（不含纯数字、不含空列）
_label_cols = []
for col in sorted(adata.obs.columns):
    if not any(kw in col for kw in ("cell_type", "_v1")):
        continue
    if col.endswith("_proportion") or col.endswith("_entropy"):
        continue
    if col == "cell_type_final_v1":
        continue  # 最终标签由 PI 拍板，不参与双向比较
    # 排除全 NaN 列
    if adata.obs[col].notna().sum() == 0:
        continue
    # 排除数值列（如 uncertainty）
    if adata.obs[col].dtype == "float64" and "uncertainty" in col:
        continue
    _label_cols.append(col)

print(f"可用标注列 ({len(_label_cols)}):")
for col in _label_cols:
    _n = adata.obs[col].nunique()
    print(f"  {col}  ({_n} 类)")
    # 确保是 category 类型
    if adata.obs[col].dtype.name != "category":
        adata.obs[col] = adata.obs[col].astype(str).astype("category")

In [ ]:
# 成对混淆矩阵热图 + Cohen's kappa
# 每个方法对出两张图：(1) 混淆矩阵热图 (2) 打印 kappa 值
from itertools import combinations

if len(_label_cols) >= 2:
    _kappa_results = []
    for _a, _b in combinations(_label_cols, 2):
        # 只比较都有有效标签的细胞
        _valid = adata.obs[_a].notna() & adata.obs[_b].notna()
        if _valid.sum() < 2:
            continue

        # ---- 混淆矩阵热图 ----
        _cm = pd.crosstab(
            adata.obs.loc[_valid, _a].astype(str),
            adata.obs.loc[_valid, _b].astype(str),
            normalize="index",
        )
        fig, ax = plt.subplots(
            figsize=(max(6, len(_cm.columns) * 1.3),
                     max(5, len(_cm.index) * 0.8))
        )
        sns.heatmap(_cm, annot=True, fmt=".2f", cmap="Blues",
                    vmin=0, vmax=1, ax=ax,
                    cbar_kws={"label": "proportion (row-normalized)"})
        ax.set_title(f"{_a}  ->  {_b}")
        ax.set_xlabel(_b)
        ax.set_ylabel(_a)
        plt.tight_layout()
        _safe_a = _a.replace("/", "_").replace(" ", "_")
        _safe_b = _b.replace("/", "_").replace(" ", "_")
        _fname = f"results/figures/06_confusion_{_safe_a}__vs__{_safe_b}.png"
        fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  混淆矩阵热图: {_fname}")
        plt.close("all")

        # ---- Cohen's kappa ----
        _k = annotation_concordance(adata, label_a=_a, label_b=_b)
        _kappa = _k.get("cohen_kappa", np.nan)
        _kappa_results.append({
            "method_a": _a, "method_b": _b,
            "cohen_kappa": round(float(_kappa), 4),
            "n_cells": int(_valid.sum()),
        })
        print(f"  {_a} vs {_b}: kappa={_kappa:.4f} (n={_valid.sum():,})")

    _kappa_df = pd.DataFrame(_kappa_results)
    if len(_kappa_df) > 0:
        print(f"\nCohen's kappa 汇总:")
        try:
            from IPython.display import display as ipy_display
            ipy_display(_kappa_df)
        except ImportError:
            print(_kappa_df)
        # 成对 kappa 写 CSV
        _kappa_df.to_csv("results/figures/06_kappa_pairs.csv", index=False)
        print("  成对 kappa 表已保存: results/figures/06_kappa_pairs.csv")
        adata.uns["cross_method_comparison_v1"] = {
            "label_columns": _label_cols,
            "kappa_csv": "results/figures/06_kappa_pairs.csv",
            "timestamp": datetime.datetime.now().isoformat(),
        }
else:
    print("可用标注列 <2，跳过成对比较")

In [ ]:
# Sankey 图——可视化两种方法之间的标签流
# 每个节点 = 一种细胞类型标签，连线宽度 = 细胞数
# 为什么用 Sankey？直观展示"方法 A 的 T_cell 被方法 B 拆成了哪些亚型"
# 流向矩阵保存为 CSV，用 R/plotly 等外部工具绘制（matplotlib.sankey 不稳健）

if len(_label_cols) >= 2:
    for _a, _b in combinations(_label_cols, 2):
        _valid = adata.obs[_a].notna() & adata.obs[_b].notna()
        if _valid.sum() < 10:
            continue
        # 构建流向表（method_a -> method_b）
        _flow = pd.crosstab(
            adata.obs.loc[_valid, _a].astype(str),
            adata.obs.loc[_valid, _b].astype(str),
        )
        _safe_a = _a.replace("/", "_").replace(" ", "_")
        _safe_b = _b.replace("/", "_").replace(" ", "_")
        _fname = f"results/figures/06_sankey/flow_{_safe_a}__vs__{_safe_b}.csv"
        _flow.to_csv(_fname)
        print(f"  流向矩阵已保存: {_fname} (shape={_flow.shape})")

    print(f"\nSankey 流向矩阵已保存至 results/figures/06_sankey/")
    print("（用 R/plotly 等工具绘制 Sankey 图）")
else:
    print("可用标注列 <2，跳过 Sankey")

## LLM 证据汇总 + 三色分级

LLM 在这里不是"第 N+1 个注释方法"——而是**智能汇总员**。

### 为什么需要 LLM 汇总而不是直接看交叉比对表？

交叉比对（混淆矩阵 + Cohen's kappa）能告诉你"方法 A 和方法 B 在某簇上不一致"，
但不能告诉你**谁更可能对**、**分歧的原因是什么**、**PI 应该怎么裁决**。

LLM 综合以下全部证据，逐簇给出结构化的判决建议：
1. **各方法标签**（marker / mLLMCelltype 共识 / 基因集评分 / scANVI ——如有）
2. **已发表专家注释**（作者原始 `cell_type_original` 列，如有）
3. **Top 10 差异表达基因**（每个簇最特异的 marker）
4. **基因集评分 profile**（该簇对各类细胞标记物的整体评分）
5. **转化状态信号**（该簇是否处于 Chief→SPEM→IM 谱系的中间态）

### 三色输出

| 颜色 | 置信度 | 含义 | PI 动作 |
|------|--------|------|---------|
| GREEN | HIGH | 多方法一致、DEG 与 marker profile 完全吻合 | 可直接接受 |
| YELLOW | MEDIUM | 部分方法有分歧，或 DEG 与预期不完全一致 | 建议查看 dotplot / UMAP 确认 |
| RED | LOW | 多方法严重分歧，或 DEG/marker profile 异常 | PI 必须手动逐基因分析 |

### 怎么看判决结果？

每簇输出一个 JSON，包含：
- `label`：建议的细胞类型标签
- `confidence`：HIGH / MEDIUM / LOW
- `reasoning`：关键判断依据（<=50 字）
- `conflict_analysis`：各方法分歧原因（如无分歧写"一致"）
- `suggestion`：给 PI 的行动建议

结果同时保存到磁盘（`results/figures/06_verdicts/verdict_summary.json`），
方便 PI 在 Jupyter 外查看。


In [ ]:
# === LLM 证据汇总 + 三色分级 ===
# 每簇收集全部证据 → 结构化 prompt → 调用 LLM（VERDICT_MODEL_TIER）→
# 解析 JSON 判决 → 输出汇总表 + 预填 pi_decisions
# 为什么不是"第 N+1 个注释方法"？LLM 在这里不做从头标注——
# 而是综合已有方法的标签、DEG、基因集评分、转化检测，给出"建议标签 + 置信度"。
# 最终判断权在 PI。

print("===== LLM 综合判决（逐簇证据汇总）=====\n")

_verdict_results = {}  # {cluster_id: {label, confidence, reasoning, color, ...}}

# 准备证据
try:
    from scrna_integration.llm_config import load_llm_group_config, get_active_groups
    _active_groups = get_active_groups(project_root=_root)
except Exception:
    _active_groups = []
_can_call_llm = len(_active_groups) > 0

if _can_call_llm:
    _cfg = load_llm_group_config(group=LLM_GROUP, project_root=_root)
    _verdict_model = _cfg.get("models", {}).get(VERDICT_MODEL_TIER, "") if _cfg else ""
    _verdict_key = _cfg.get("api_key", "") if _cfg else ""
    _verdict_url = _cfg.get("base_url", "") if _cfg else ""
    _verdict_provider = _cfg.get("provider", "") if _cfg else ""

# 导入 json（用于解析 LLM 返回的 JSON 判决）
import json as _json

if _can_call_llm and _verdict_model:
    # 确保有 rank_genes_groups
    if "rank_genes_06" not in adata.uns:
        sc.tl.rank_genes_groups(
            adata, groupby=LEIDEN_COL, method="wilcoxon",
            n_genes=30, key_added="rank_genes_06",
        )

    # 复用 cell 22 已收集的 _label_cols（如不存在则重新收集）
    if "_label_cols" not in dir() or not _label_cols:
        _label_cols = [col for col in sorted(adata.obs.columns)
                       if any(kw in col for kw in ("cell_type", "_v1"))
                       and not any(skip in col for skip in ("proportion", "entropy", "uncertainty", "final"))
                       and adata.obs[col].notna().any()]

    # 转化检测结果（从 adata.uns 读取，不依赖 cell 间变量）
    _transition_flags = adata.uns.get("transition_detection_v1", {})

    for cl in sorted(adata.obs[LEIDEN_COL].unique(), key=lambda x: int(x)):
        _mask = adata.obs[LEIDEN_COL] == cl

        # ==== 收集证据 ====
        _evidence = f"## Cluster {cl} ({int(_mask.sum())} cells)\n\n"

        # 证据 1: 各方法标签
        _evidence += "### 各方法注释结果\n"
        for col in _label_cols:
            _cl_labels = adata.obs.loc[_mask, col].dropna()
            if len(_cl_labels) > 0:
                _top = _cl_labels.mode().iloc[0]
                _pct = (_cl_labels == _top).mean()
                _evidence += f"- {col}: {_top} ({_pct:.0%})\n"

        # 证据 2: 作者原始注释（如有）
        _ref_col = None
        for cand in ["cell_type_original", "Celltypes_global"]:
            if cand in adata.obs.columns:
                _ref_col = cand
                break
        if _ref_col:
            _ref_labels = adata.obs.loc[_mask, _ref_col].dropna()
            if len(_ref_labels) > 0:
                _ref_top = _ref_labels.mode().iloc[0]
                _ref_pct = (_ref_labels == _ref_top).mean()
                _evidence += f"- **已发表专家注释**({_ref_col}): {_ref_top} ({_ref_pct:.0%})\n"

        # 证据 3: Top DEG
        if "rank_genes_06" in adata.uns:
            _genes = []
            try:
                _df = sc.get.rank_genes_groups_df(adata, group=str(cl), key="rank_genes_06")
                _genes = _df["names"].head(10).tolist()
            except Exception:
                pass
            if _genes:
                _evidence += f"\n### Top 10 DEG\n{', '.join(_genes[:10])}\n"

        # 证据 4: 基因集评分
        _score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
        if _score_cols:
            _evidence += "\n### 基因集评分（簇均值）\n"
            _scores_this = {
                c.replace("score_", ""): f"{adata.obs.loc[_mask, c].mean():.3f}"
                for c in _score_cols
            }
            _sorted_scores = sorted(_scores_this.items(), key=lambda x: -float(x[1]))[:5]
            for name, val in _sorted_scores:
                _evidence += f"- {name}: {val}\n"

        # 证据 5: 转化状态（如检测到）
        if str(cl) in _transition_flags:
            _tf = _transition_flags[str(cl)]
            _evidence += "\n### ⚠️ 转化状态信号\n"
            _evidence += f"- 检测到 {_tf['axis']} 转化（{_tf['stage']}，进度 {_tf['progress']}）\n"
            _evidence += f"- declining markers 均值: {_tf['declining_mean']}, rising markers 均值: {_tf['rising_mean']}\n"
            _evidence += "- 建议：标注为转化中间态而非单一离散类型\n"

        # ==== 构建 prompt ====
        _system = (
            "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱。"
            "请综合以下多维证据，判断该 cluster 的细胞类型。"
        )
        _user = f"""{_evidence}

请以 JSON 格式返回（不要 markdown 围栏）：
{{
  "label": "最可能的细胞类型（具体到亚型）",
  "confidence": "HIGH/MEDIUM/LOW",
  "reasoning": "50字内关键理由",
  "conflict_analysis": "各方法分歧原因（如无分歧写'一致'）",
  "suggestion": "给 PI 的建议（如'可直接接受'或'建议看 dotplot 确认 XX 表达'）"
}}"""

        # ==== 调用 LLM ====
        try:
            if _verdict_provider == "anthropic":
                import requests as _req
                _resp = _req.post(
                    f"{_verdict_url}/v1/messages",
                    headers={
                        "x-api-key": _verdict_key,
                        "anthropic-version": "2023-06-01",
                        "content-type": "application/json",
                    },
                    json={
                        "model": _verdict_model,
                        "max_tokens": 800,
                        "system": _system,
                        "messages": [{"role": "user", "content": _user}],
                    },
                    timeout=60,
                )
                _data = _resp.json()
                _text_blocks = [
                    b.get("text", "")
                    for b in _data.get("content", [])
                    if b.get("type") == "text"
                ]
                _raw = "\n".join(_text_blocks)
            else:
                # OpenAI-compatible
                from openai import OpenAI
                _client = OpenAI(api_key=_verdict_key, base_url=_verdict_url)
                _completion = _client.chat.completions.create(
                    model=_verdict_model,
                    max_tokens=800,
                    temperature=0.3,
                    messages=[
                        {"role": "system", "content": _system},
                        {"role": "user", "content": _user},
                    ],
                )
                _raw = _completion.choices[0].message.content

            # 解析 JSON
            _raw_clean = _raw.strip()
            if _raw_clean.startswith("```"):
                _lines = _raw_clean.split("\n")
                _raw_clean = "\n".join(_lines[1:-1]) if len(_lines) >= 3 else _raw_clean
            _parsed = _json.loads(_raw_clean)
            _verdict_results[str(cl)] = _parsed

            _color_map = {"HIGH": "[GREEN]", "MEDIUM": "[YELLOW]", "LOW": "[RED]"}
            _color = _color_map.get(_parsed.get("confidence", ""), "[WHITE]")
            print(f"  {_color} Cluster {cl}: {_parsed.get('label', '?')} "
                  f"({_parsed.get('confidence', '?')}) -- {_parsed.get('reasoning', '')}")

        except Exception as e:
            print(f"  ⚠️ Cluster {cl} 判决失败: {e}")
            _verdict_results[str(cl)] = {
                "label": "",
                "confidence": "LOW",
                "reasoning": f"LLM 调用失败: {e}",
                "conflict_analysis": "",
                "suggestion": "需 PI 手动判断",
            }

    # 保存汇总 JSON
    os.makedirs("results/figures/06_verdicts", exist_ok=True)
    with open("results/figures/06_verdicts/verdict_summary.json", "w") as f:
        _json.dump(_verdict_results, f, ensure_ascii=False, indent=2)
    print("\n判决汇总已保存: results/figures/06_verdicts/verdict_summary.json")

    # 统计三色分布
    _color_counts = {"HIGH": 0, "MEDIUM": 0, "LOW": 0}
    for v in _verdict_results.values():
        _c = v.get("confidence", "")
        if _c in _color_counts:
            _color_counts[_c] += 1
    print(f"三色分布: [GREEN]-HIGH={_color_counts['HIGH']}, "
          f"[YELLOW]-MEDIUM={_color_counts['MEDIUM']}, "
          f"[RED]-LOW={_color_counts['LOW']}")
    print("  -> [GREEN] 簇可直接接受，[YELLOW]/[RED] 请 PI 重点复核")

else:
    print("⚠️ 无可用 LLM 配置（.env 缺少 LLM_GROUP 或 VERDICT_MODEL_TIER 档模型），跳过证据汇总")
    print("  -> fallback: 用 claude -p 手动逐簇分析")
    _verdict_results = {}


## PI 拍板：`cell_type_final_v1`

PI 阅读每簇的 LLM 判决 markdown 后，手动决定最终标签。
**为什么不让 LLM 自动拍板？** 每个簇的最终标签是科学判断，
LLM 是顾问不是决策者——PI 结合自身领域知识复核后决定。
对胃癌前病变项目（~20-30 簇），阅读所有判决约需 30-60 分钟。

In [ ]:
# === PI 拍板（LLM 建议预填）===
# [GREEN] HIGH 置信度的簇已自动预填——PI 可直接接受或覆盖
# [YELLOW] MEDIUM / [RED] LOW 的簇留空——需 PI 手动判断
# 为什么还要 PI 拍板？LLM 是顾问，不是决策者。
# 转化中间态、亚型边界、临床意义——这些判断需要 PI 的领域知识。

pi_decisions = {}
if "_verdict_results" in dir() and _verdict_results:
    for cl, v in _verdict_results.items():
        if v.get("confidence") == "HIGH" and v.get("label"):
            pi_decisions[cl] = v["label"]
        else:
            pi_decisions[cl] = ""  # 留空让 PI 手动填入

    print("===== PI 拍板区 =====")
    print("（[GREEN] 已预填，[YELLOW]/[RED] 需 PI 手动填入标签）\n")
    _color_map = {"HIGH": "[GREEN]", "MEDIUM": "[YELLOW]", "LOW": "[RED]"}
    for cl in sorted(pi_decisions.keys(), key=lambda x: int(x)):
        _v = _verdict_results.get(cl, {})
        _color = _color_map.get(_v.get("confidence", ""), "[WHITE]")
        if pi_decisions[cl]:
            _label = pi_decisions[cl]
            print(f'    "{cl}": "{_label}",  # {_color} LLM 预填: {_v.get("reasoning", "")}')
        else:
            print(f'    "{cl}": "",  # {_color} <- PI 填入  {_v.get("reasoning", "")}')
else:
    print("无 LLM 判决结果（证据汇总未运行或失败），PI 需从头手动标注所有簇")
    # 预生成空字典供 PI 逐簇填写
    for cl in sorted(adata.obs[LEIDEN_COL].unique(), key=lambda x: int(x)):
        pi_decisions[str(cl)] = ""

# 应用 PI 拍板结果到 adata
if pi_decisions:
    # 过滤掉空值（PI 还未填的簇）
    _filled = {k: v for k, v in pi_decisions.items() if v}
    if _filled:
        adata.obs["cell_type_final_v1"] = (
            adata.obs[LEIDEN_COL].astype(str).map(_filled).astype("category")
        )
        n_final = adata.obs["cell_type_final_v1"].notna().sum()
        print(f"\n最终标注: {n_final}/{adata.n_obs} 细胞已标注 ({len(_filled)} 簇)")
        print(f"  标签种类: {adata.obs['cell_type_final_v1'].nunique()}")
    else:
        adata.obs["cell_type_final_v1"] = pd.Categorical([np.nan] * adata.n_obs)
        print("\nPI 暂未填入任何标签，已预建 cell_type_final_v1 空列")
else:
    adata.obs["cell_type_final_v1"] = pd.Categorical([np.nan] * adata.n_obs)
    print("PI 暂未拍板，已预建 cell_type_final_v1 空列")


In [ ]:
# 记录 PI 拍板的元数据——plain adata.uns 写入，PI 自由记录
adata.uns["cell_type_final_v1_notes"] = {
    "leiden_resolution_used": LEIDEN_COL,
    "available_methods": _label_cols if "_label_cols" in dir() else [],
    "method_basis": "PI manual review of LLM verdicts + marker dotplot",
    "rationale": "PI reviewed each cluster verdict; final labels reflect domain expertise",
    "timestamp": datetime.datetime.now().isoformat(),
}
print("PI 拍板元数据已写入 adata.uns['cell_type_final_v1_notes']")

In [ ]:
# 运行追踪字段——stage + version + upstream
adata.uns["stage"] = "06_annotated"
adata.uns["version"] = "v1"
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"

# 记录 06 运行元数据
adata.uns["06_annotated_v1"] = {
    "upstream": UPSTREAM_PATH,
    "leiden_col": LEIDEN_COL,
    "marker_csv": MARKER_CSV,
    "methods_run": [
        "marker_dotplot",
        "mllmcelltype_consensus",
        "gene_set_scoring",
        "transition_detection",
    ],
    "scANVI_skipped": not bool(REFERENCE_ATLAS_PATH),
    "timestamp": datetime.datetime.now().isoformat(),
}
print("06 运行元数据已记录")


In [ ]:
# 写入前自检——守卫 adata.X 不变
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("自检通过: X 是 sparse CSR float32")

In [ ]:
# 写出 stage checkpoint（lzf 压缩以节省磁盘空间）
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")

assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 跨 stage 边界释放内存
del adata
gc.collect()
print("内存已释放")